# Confident AI Tracing — All Scripts

Each section below corresponds to one of the original Python files.
Set your environment variables in the **⚙️ Setup** cell before running any other cell.

| # | Cell | File | Purpose |
|---|------|------|---------|
| 1 | Setup | — | Install deps & set env vars |
| 2 | `confident_ai_tracing.py` | Basic RAG trace (1 retriever + 1 LLM span) |
| 3 | `confident_tracing2.py` | RAG trace — context logged inside LLM span |
| 4 | `tracing_filterred.py` | RAG trace — extra params filtered before logging |
| 5 | `pull_trace_time.py` | Pull all traces in a time window |
| 6 | `trace_by_input.py` | Find most-recent trace by input text |
| 7 | `list_spans.py` | List spans with server-side filters |
| 8 | `evaluate_trace.py` | Pull a trace and run AnswerRelevancy metric |
| 9 | `span_filter_eval.py` | Pull LLM span and run Summarization / Faithfulness metric |

---
## ⚙️ Cell 1 — Setup: Install dependencies & configure secrets

In [ ]:
# ---- Install dependencies ----
!pip install -q -U deepeval openai python-dotenv

import os

# ---- Set your credentials here (or load from a .env file) ----
os.environ["CONFIDENT_API_KEY"]          = "YOUR_CONFIDENT_API_KEY"
os.environ["AZURE_OPENAI_API_KEY"]       = "YOUR_AZURE_OPENAI_API_KEY"
os.environ["AZURE_OPENAI_ENDPOINT"]      = "https://<your-resource>.openai.azure.com/"
os.environ["AZURE_OPENAI_API_VERSION"]   = "2024-10-21"
os.environ["AZURE_OPENAI_DEPLOYMENT"]    = "your-deployment-name"
os.environ["AZURE_OPENAI_MODEL"]         = "gpt-4o-mini"

# Optional: set these only if needed by tracing_filterred.py
os.environ.setdefault("VECTOR_INDEX_KEY", "secret-index-key")

print("✅ Environment configured.")

---
## 📄 Cell 2 — `confident_ai_tracing.py`
Minimal Confident AI trace: **1 retriever span + 1 LLM span**.
The LLM span logs only the question as its input.

In [ ]:
"""
Minimal Confident AI trace: 1 retriever + 1 LLM span.

NOTE: This file DISABLES SSL verification globally. That is acceptable for
dev / debugging on a corporate machine where SSL-inspection is breaking
cert validation. For production: install your corporate root CA into
certifi's bundle and REMOVE the patch_ssl() block below.
"""

import os
import ssl
import warnings


# ---- Disable SSL verification globally (corp-proxy workaround) ----
def patch_ssl():
    import urllib3
    import requests

    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    warnings.filterwarnings("ignore", message="Unverified HTTPS request")
    ssl._create_default_https_context = ssl._create_unverified_context

    _orig_send = requests.Session.send

    def _send_no_verify(self, request, **kwargs):
        kwargs["verify"] = False
        return _orig_send(self, request, **kwargs)

    requests.Session.send = _send_no_verify


patch_ssl()

os.environ["CONFIDENT_TRACE_FLUSH"] = "1"
os.environ.setdefault("CONFIDENT_TRACE_VERBOSE", "0")

from openai import AzureOpenAI
from deepeval.tracing import (
    observe,
    update_current_trace,
    update_current_span,
    update_llm_span,
)

client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
)
AZURE_DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]
UNDERLYING_MODEL = os.environ.get("AZURE_OPENAI_MODEL", "gpt-4o-mini")


# ---------- RETRIEVER SPAN ----------
@observe(type="retriever", embedder="text-embedding-ada-002")
def retrieve_context(query: str) -> list[str]:
    kb = [
        "The Eiffel Tower is located in Paris, France.",
        "It was completed in 1889 and stands 330 meters tall.",
        "The Eiffel Tower was designed by Gustave Eiffel's company.",
        "Python is a high-level programming language created by Guido van Rossum.",
    ]
    chunks = [doc for doc in kb if any(w in doc.lower() for w in query.lower().split())]
    update_current_span(input=query, output=chunks)
    return chunks


# ---------- LLM SPAN ----------
@observe(
    type="llm",
    model=UNDERLYING_MODEL,
    cost_per_input_token=0.00000015,
    cost_per_output_token=0.0000006,
)
def call_llm(question: str, context: list[str]) -> str:
    messages = [
        {"role": "system",
         "content": "Answer using ONLY the context below.\n\nContext:\n" + "\n".join(context)},
        {"role": "user", "content": question},
    ]
    resp = client.chat.completions.create(model=AZURE_DEPLOYMENT, messages=messages)
    answer = resp.choices[0].message.content

    update_current_span(input=question, output=answer)
    update_llm_span(
        input_token_count=resp.usage.prompt_tokens,
        output_token_count=resp.usage.completion_tokens,
    )
    return answer


# ---------- THE TRACE ----------
@observe()
def rag_qa(question: str) -> str:
    context = retrieve_context(question)
    answer = call_llm(question, context)
    update_current_trace(input=question, output=answer)
    return answer


# ---- Run ----
question = "Who designed the Eiffel Tower and when was it built?"
print("Q:", question)
print("A:", rag_qa(question))
print("\n✅ View your trace at https://app.confident-ai.com")

---
## 📄 Cell 3 — `confident_tracing2.py`
Same RAG trace, but the LLM span logs **both the question and the retrieved context** as its input dict, so downstream evaluators can read context straight from the span.

In [ ]:
"""
Minimal Confident AI trace: 1 retriever + 1 LLM span.
The LLM span logs BOTH the question and the retrieved context as its input,
so downstream evaluators can read context straight from the LLM span.
"""

import os
import ssl
import warnings


def patch_ssl():
    import urllib3
    import requests
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    warnings.filterwarnings("ignore", message="Unverified HTTPS request")
    ssl._create_default_https_context = ssl._create_unverified_context
    _orig_send = requests.Session.send

    def _send_no_verify(self, request, **kwargs):
        kwargs["verify"] = False
        return _orig_send(self, request, **kwargs)
    requests.Session.send = _send_no_verify


patch_ssl()
os.environ["CONFIDENT_TRACE_FLUSH"] = "1"
os.environ.setdefault("CONFIDENT_TRACE_VERBOSE", "0")

from openai import AzureOpenAI
from deepeval.tracing import (
    observe,
    update_current_trace,
    update_current_span,
    update_llm_span,
)

client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
)
AZURE_DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]
UNDERLYING_MODEL = os.environ.get("AZURE_OPENAI_MODEL", "gpt-4o-mini")


# ---------- RETRIEVER SPAN ----------
@observe(type="retriever", embedder="text-embedding-ada-002")
def retrieve_context(query: str) -> list[str]:
    kb = [
        "The Eiffel Tower is located in Paris, France.",
        "It was completed in 1889 and stands 330 meters tall.",
        "The Eiffel Tower was designed by Gustave Eiffel's company.",
        "Python is a high-level programming language created by Guido van Rossum.",
    ]
    chunks = [doc for doc in kb if any(w in doc.lower() for w in query.lower().split())]
    update_current_span(input=query, output=chunks)
    return chunks


# ---------- LLM SPAN ----------
# Input logged as a dict so context is preserved on the span itself.
@observe(
    type="llm",
    model=UNDERLYING_MODEL,
    cost_per_input_token=0.00000015,
    cost_per_output_token=0.0000006,
)
def call_llm(question: str, context: list[str]) -> str:
    messages = [
        {"role": "system",
         "content": "Answer using ONLY the context below.\n\nContext:\n" + "\n".join(context)},
        {"role": "user", "content": question},
    ]
    resp = client.chat.completions.create(model=AZURE_DEPLOYMENT, messages=messages)
    answer = resp.choices[0].message.content

    # Log BOTH question and context as the LLM span's input
    update_current_span(
        input={"question": question, "context": context},
        output=answer,
    )
    update_llm_span(
        input_token_count=resp.usage.prompt_tokens,
        output_token_count=resp.usage.completion_tokens,
    )
    return answer


# ---------- THE TRACE ----------
@observe()
def rag_qa(question: str) -> str:
    context = retrieve_context(question)
    answer = call_llm(question, context)
    update_current_trace(input=question, output=answer)
    update_current_span(input=question, output=answer)
    return answer


# ---- Run ----
question = "Who designed the Eiffel Tower and when was it built?"
print("Q:", question)
print("A:", rag_qa(question))
print("\n✅ View your trace at https://app.confident-ai.com")

---
## 📄 Cell 4 — `tracing_filterred.py`
RAG demo where functions have **extra internal parameters** (API keys, user IDs, etc.).
We override what's logged so only the relevant fields appear in the Confident AI UI.

In [ ]:
"""
RAG demo where each function has extra parameters and we override what's
logged to Confident AI so only the relevant fields show up in the UI.

Critical: for the outermost function (rag_qa) we override BOTH trace-level
AND span-level input/output — they are separate.
"""

import os
import ssl
import warnings


def patch_ssl():
    import urllib3
    import requests
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    warnings.filterwarnings("ignore", message="Unverified HTTPS request")
    ssl._create_default_https_context = ssl._create_unverified_context
    _orig_send = requests.Session.send

    def _send_no_verify(self, request, **kwargs):
        kwargs["verify"] = False
        return _orig_send(self, request, **kwargs)
    requests.Session.send = _send_no_verify


patch_ssl()
os.environ["CONFIDENT_TRACE_FLUSH"] = "1"
os.environ.setdefault("CONFIDENT_TRACE_VERBOSE", "0")

from openai import AzureOpenAI
from deepeval.tracing import (
    observe,
    update_current_trace,
    update_current_span,
    update_llm_span,
)

client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
)
AZURE_DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]
UNDERLYING_MODEL = os.environ.get("AZURE_OPENAI_MODEL", "gpt-4o-mini")


@observe(type="retriever", embedder="text-embedding-ada-002")
def retrieve_context(query: str, top_k: int, namespace: str, index_key: str) -> list[str]:
    kb = [
        "The Eiffel Tower is located in Paris, France.",
        "It was completed in 1889 and stands 330 meters tall.",
        "The Eiffel Tower was designed by Gustave Eiffel's company.",
        "Python is a high-level programming language created by Guido van Rossum.",
    ]
    chunks = [doc for doc in kb if any(w in doc.lower() for w in query.lower().split())][:top_k]
    update_current_span(input=query, output=chunks)
    return chunks


@observe(
    type="llm",
    model=UNDERLYING_MODEL,
    cost_per_input_token=0.00000015,
    cost_per_output_token=0.0000006,
)
def call_llm(
    question: str,
    context: list[str],
    temperature: float,
    max_tokens: int,
    api_key: str,
    user_id: str,
) -> str:
    messages = [
        {"role": "system",
         "content": "Answer using ONLY the context below.\n\nContext:\n" + "\n".join(context)},
        {"role": "user", "content": question},
    ]
    resp = client.chat.completions.create(
        model=AZURE_DEPLOYMENT,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    answer = resp.choices[0].message.content

    update_current_span(
        input={"question": question, "context": context},
        output=answer,
    )
    update_llm_span(
        input_token_count=resp.usage.prompt_tokens,
        output_token_count=resp.usage.completion_tokens,
    )
    return answer


@observe()
def rag_qa(
    question: str,
    user_id: str,
    session_id: str,
    request_metadata: dict,
) -> str:
    context = retrieve_context(
        query=question,
        top_k=3,
        namespace="prod-docs",
        index_key=os.environ.get("VECTOR_INDEX_KEY", "secret-index-key"),
    )
    answer = call_llm(
        question=question,
        context=context,
        temperature=0.2,
        max_tokens=512,
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        user_id=user_id,
    )

    # Override at BOTH levels — they are independent.
    # 1. Trace-level (visible in traces list, used by evaluate_trace)
    update_current_trace(input=question, output=answer)
    # 2. Span-level (visible when this span is clicked in the tree)
    update_current_span(input=question, output=answer)

    return answer


# ---- Run ----
question = "Who designed the Eiffel Tower and when was it built?"
print("Q:", question)
answer = rag_qa(
    question=question,
    user_id="user-12345",
    session_id="sess-abcdef",
    request_metadata={"ip": "10.0.0.1", "trace_id_from_caller": "xyz"},
)
print("A:", answer)
print("\n✅ View your trace at https://app.confident-ai.com")

---
## 📄 Cell 5 — `pull_trace_time.py`
Pull **all traces in a time window** and display them with all child spans.
Edit `HOURS` to change the lookback window.

In [ ]:
"""
Pull traces from Confident AI in a time window and display them, including
all child spans (retriever, LLM, tool, etc.).

Edit HOURS below to change the lookback window.
"""

import os
import ssl
import requests
import urllib3
from datetime import datetime, timedelta, timezone

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
ssl._create_default_https_context = ssl._create_unverified_context
_orig_send = requests.Session.send
requests.Session.send = lambda self, r, **kw: _orig_send(self, r, **{**kw, "verify": False})

API = "https://api.confident-ai.com/v1"
HEADERS = {"CONFIDENT_API_KEY": os.environ["CONFIDENT_API_KEY"]}

# ==== Edit this to set the lookback window ====
HOURS = 1      # e.g. 1 = last hour, 24 = last day, 168 = last 7 days
# ==============================================


def list_traces(start: datetime, end: datetime) -> list[dict]:
    """Page through all traces in [start, end]."""
    traces, cursor = [], None
    while True:
        params = {
            "start": start.isoformat().replace("+00:00", "Z"),
            "end": end.isoformat().replace("+00:00", "Z"),
            "pageSize": 100,
        }
        if cursor:
            params["cursor"] = cursor
        r = requests.get(f"{API}/traces", headers=HEADERS, params=params, verify=False, timeout=30)
        r.raise_for_status()
        data = r.json()["data"]
        traces.extend(data.get("traces", []))
        cursor = data.get("nextCursor")
        if not cursor:
            break
    return traces


def fetch_trace(uuid: str) -> dict:
    r = requests.get(f"{API}/traces/{uuid}", headers=HEADERS, verify=False, timeout=30)
    r.raise_for_status()
    return r.json()["data"]


# ---- Run ----
end = datetime.now(timezone.utc)
start = end - timedelta(hours=HOURS)

print(f"Window : {start.isoformat()}  ->  {end.isoformat()}  (last {HOURS}h)\n")

summaries = list_traces(start, end)
print(f"Found {len(summaries)} trace(s)\n")

for i, summary in enumerate(summaries, 1):
    trace = fetch_trace(summary["uuid"])
    print("=" * 70)
    print(f"[{i}/{len(summaries)}] TRACE {trace['uuid']}")
    print(f"  name        : {trace.get('name')}")
    print(f"  environment : {trace.get('environment')}")
    print(f"  start       : {trace.get('startTime')}")
    print(f"  input       : {trace.get('input')}")
    print(f"  output      : {trace.get('output')}")

    spans = trace.get("spans") or []
    print(f"\n  SPANS ({len(spans)}):")
    for j, span in enumerate(spans, 1):
        print(f"  --- span {j} ---")
        print(f"    uuid    : {span.get('uuid')}")
        print(f"    name    : {span.get('name')}")
        print(f"    type    : {span.get('type')}")
        print(f"    input   : {span.get('input')}")
        print(f"    output  : {span.get('output')}")
        if span.get("type") == "LLM":
            print(f"    model   : {span.get('model')}")
            print(f"    tokens  : in={span.get('inputTokenCount')} out={span.get('outputTokenCount')}")
        if span.get("type") == "RETRIEVER":
            print(f"    embedder: {span.get('embedder')}")
    print()

---
## 📄 Cell 6 — `trace_by_input.py`
Find the **most recent trace whose trace-level input matches** `TARGET_INPUT`.
Edit `TARGET_INPUT` and `HOURS` below.

In [ ]:
"""
Find the most recent trace whose trace-level input matches TARGET_INPUT,
then display the full trace + all child spans.

Edit TARGET_INPUT and HOURS below.
"""

import os
import ssl
import requests
import urllib3
from datetime import datetime, timedelta, timezone

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
ssl._create_default_https_context = ssl._create_unverified_context
_orig_send = requests.Session.send
requests.Session.send = lambda self, r, **kw: _orig_send(self, r, **{**kw, "verify": False})

API = "https://api.confident-ai.com/v1"
HEADERS = {"CONFIDENT_API_KEY": os.environ["CONFIDENT_API_KEY"]}

# ==== Edit these ====
TARGET_INPUT = "Who designed the leaning tower of Piza?"
HOURS = 5   # how far back to search (168h = 7 days)
# ====================


def list_traces(start: datetime, end: datetime) -> list[dict]:
    """Page through all trace summaries in [start, end]."""
    traces, cursor = [], None
    while True:
        params = {
            "start": start.isoformat().replace("+00:00", "Z"),
            "end": end.isoformat().replace("+00:00", "Z"),
            "pageSize": 100,
        }
        if cursor:
            params["cursor"] = cursor
        r = requests.get(f"{API}/traces", headers=HEADERS, params=params, verify=False, timeout=30)
        r.raise_for_status()
        data = r.json()["data"]
        traces.extend(data.get("traces", []))
        cursor = data.get("nextCursor")
        if not cursor:
            break
    return traces


def fetch_trace(uuid: str) -> dict:
    r = requests.get(f"{API}/traces/{uuid}", headers=HEADERS, verify=False, timeout=30)
    r.raise_for_status()
    return r.json()["data"]


def print_trace(trace: dict) -> None:
    print("=" * 70)
    print(f"TRACE {trace['uuid']}")
    print(f"  name        : {trace.get('name')}")
    print(f"  environment : {trace.get('environment')}")
    print(f"  start       : {trace.get('startTime')}")
    print(f"  input       : {trace.get('input')}")
    print(f"  output      : {trace.get('output')}")

    spans = trace.get("spans") or []
    print(f"\n  SPANS ({len(spans)}):")
    for j, span in enumerate(spans, 1):
        print(f"  --- span {j} ---")
        print(f"    uuid    : {span.get('uuid')}")
        print(f"    name    : {span.get('name')}")
        print(f"    type    : {span.get('type')}")
        print(f"    input   : {span.get('input')}")
        print(f"    output  : {span.get('output')}")
        if span.get("type") == "LLM":
            print(f"    model   : {span.get('model')}")
            print(f"    tokens  : in={span.get('inputTokenCount')} out={span.get('outputTokenCount')}")
        if span.get("type") == "RETRIEVER":
            print(f"    embedder: {span.get('embedder')}")


# ---- Run ----
end = datetime.now(timezone.utc)
start = end - timedelta(hours=HOURS)

print(f"Searching last {HOURS}h for traces with input:\n  '{TARGET_INPUT}'\n")

summaries = list_traces(start, end)
print(f"Scanning {len(summaries)} trace(s) in window ...")

matches = []
for s in summaries:
    full = fetch_trace(s["uuid"])
    if full.get("input") == TARGET_INPUT:
        matches.append(full)

if not matches:
    print("\nNo matching traces found.")
else:
    matches.sort(key=lambda t: t.get("startTime") or "", reverse=True)
    print(f"Found {len(matches)} match(es). Showing most recent:\n")
    print_trace(matches[0])

---
## 📄 Cell 7 — `list_spans.py`
List spans using the `/v1/spans` endpoint with **server-side filters** (no trace iteration).
Edit `HOURS`, `SPAN_NAME`, and `SPAN_TYPE` below.

In [ ]:
"""
List spans from Confident AI using the official /v1/spans endpoint.
No trace iteration, no name string-matching — server-side filters.

Edit HOURS, SPAN_NAME, SPAN_TYPE below.
"""

import os
import ssl
import requests
import urllib3
from datetime import datetime, timedelta, timezone

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
ssl._create_default_https_context = ssl._create_unverified_context
_orig_send = requests.Session.send
requests.Session.send = lambda self, r, **kw: _orig_send(self, r, **{**kw, "verify": False})

API = "https://api.confident-ai.com/v1"
HEADERS = {"CONFIDENT_API_KEY": os.environ["CONFIDENT_API_KEY"]}

# ==== Filters ====
HOURS = 1
SPAN_NAME = "call_llm"      # exact match on span name (None to skip)
SPAN_TYPE = "LLM"            # one of: SPAN, AGENT, TOOL, RETRIEVER, LLM (None to skip)
INCLUDE_FULL_DATA = True     # if True, fetch each span's full input/output
# =================


def list_spans(start: datetime, end: datetime, name, type_) -> list[dict]:
    """Page through all spans matching the filters."""
    spans, page = [], 1
    while True:
        params = {
            "start": start.isoformat().replace("+00:00", "Z"),
            "end": end.isoformat().replace("+00:00", "Z"),
            "page": page,
            "pageSize": 100,
        }
        if name:
            params["name"] = name
        if type_:
            params["type"] = type_

        r = requests.get(f"{API}/spans", headers=HEADERS, params=params, verify=False, timeout=30)
        r.raise_for_status()
        data = r.json()["data"]
        batch = data.get("spans", [])
        spans.extend(batch)
        if len(batch) < params["pageSize"]:
            break
        page += 1
    return spans


def fetch_span(uuid: str) -> dict:
    """Get the full span including input/output."""
    r = requests.get(f"{API}/spans/{uuid}", headers=HEADERS, verify=False, timeout=30)
    r.raise_for_status()
    return r.json()["data"]


# ---- Run ----
end = datetime.now(timezone.utc)
start = end - timedelta(hours=HOURS)

print(f"Window    : last {HOURS}h")
print(f"Name      : {SPAN_NAME}")
print(f"Type      : {SPAN_TYPE}")

summaries = list_spans(start, end, SPAN_NAME, SPAN_TYPE)
print(f"\nFound {len(summaries)} span(s) matching filters\n")

for i, s in enumerate(summaries, 1):
    print("=" * 70)
    print(f"[{i}/{len(summaries)}] span {s.get('uuid')}")
    print(f"  name      : {s.get('name')}")
    print(f"  type      : {s.get('type')}")
    print(f"  traceUuid : {s.get('traceUuid')}")
    print(f"  startTime : {s.get('startTime')}")
    print(f"  status    : {s.get('status')}")
    if s.get("model"):
        print(f"  model     : {s.get('model')}")

    if INCLUDE_FULL_DATA:
        full = fetch_span(s["uuid"])
        print(f"  input     : {full.get('input')}")
        print(f"  output    : {full.get('output')}")

---
## 📄 Cell 8 — `evaluate_trace.py`
Pull a trace from Confident AI and evaluate it locally with **AnswerRelevancyMetric**.
Set `TRACE_UUID` or pass it as an argument.

In [ ]:
"""
Pull a trace from Confident AI and evaluate it locally with deepeval
using Azure OpenAI as the judge. Prints the metric score in the terminal.
"""

import os
import ssl
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
ssl._create_default_https_context = ssl._create_unverified_context
_orig_send = requests.Session.send
requests.Session.send = lambda self, r, **kw: _orig_send(self, r, **{**kw, "verify": False})

from deepeval.models import AzureOpenAIModel
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

# ==== Set the trace UUID to evaluate ====
TRACE_UUID = "1dcd2e83-1f74-4387-9366-9b04d3a91355"   # ← replace with your trace UUID
# =========================================


def fetch_trace(trace_uuid: str) -> dict:
    r = requests.get(
        f"https://api.confident-ai.com/v1/traces/{trace_uuid}",
        headers={"CONFIDENT_API_KEY": os.environ["CONFIDENT_API_KEY"]},
        verify=False,
        timeout=30,
    )
    r.raise_for_status()
    return r.json()["data"]


def build_judge() -> AzureOpenAIModel:
    return AzureOpenAIModel(
        model=os.environ.get("AZURE_OPENAI_MODEL", "gpt-4o-mini"),
        deployment_name=os.environ["AZURE_OPENAI_DEPLOYMENT"],
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
        base_url=os.environ["AZURE_OPENAI_ENDPOINT"],
        temperature=0,
    )


# ---- Run ----
trace = fetch_trace(TRACE_UUID)
user_input = str(trace["input"])
ai_output = str(trace["output"])

print(f"Trace  : {TRACE_UUID}")
print(f"Input  : {user_input}")
print(f"Output : {ai_output}")

metric = AnswerRelevancyMetric(model=build_judge(), threshold=0.5, include_reason=True)
metric.measure(LLMTestCase(input=user_input, actual_output=ai_output))

print(f"\nScore  : {metric.score:.3f}  (threshold {metric.threshold})")
print(f"Passed : {metric.score >= metric.threshold}")
print(f"Reason : {metric.reason}")

---
## 📄 Cell 9 — `span_filter_eval.py`
Pull a trace, extract its LLM span, and evaluate with either **SummarizationMetric** or **FaithfulnessMetric**.
Flip `METRIC` between `"summarization"` and `"faithfulness"`.

In [ ]:
"""
Pull a trace, extract its LLM span, and evaluate locally with deepeval.

Flip METRIC below between "summarization" and "faithfulness":
  - SummarizationMetric: alignment + coverage between source and "summary"
  - FaithfulnessMetric : RAG-specific, checks the answer is grounded in context
"""

import os
import ssl
import json
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
ssl._create_default_https_context = ssl._create_unverified_context
_orig_send = requests.Session.send
requests.Session.send = lambda self, r, **kw: _orig_send(self, r, **{**kw, "verify": False})

from deepeval.models import AzureOpenAIModel
from deepeval.metrics import SummarizationMetric, FaithfulnessMetric
from deepeval.test_case import LLMTestCase

# ==== Edit these ====
TRACE_UUID = "4c93866b-2dca-443f-88bc-3f3271dc031c"   # ← replace with your trace UUID
METRIC = "summarization"   # "summarization" or "faithfulness"
# ====================


def fetch_trace(trace_uuid: str) -> dict:
    r = requests.get(
        f"https://api.confident-ai.com/v1/traces/{trace_uuid}",
        headers={"CONFIDENT_API_KEY": os.environ["CONFIDENT_API_KEY"]},
        verify=False,
        timeout=30,
    )
    r.raise_for_status()
    return r.json()["data"]


def build_judge() -> AzureOpenAIModel:
    return AzureOpenAIModel(
        model=os.environ.get("AZURE_OPENAI_MODEL", "gpt-4o-mini"),
        deployment_name=os.environ["AZURE_OPENAI_DEPLOYMENT"],
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
        base_url=os.environ["AZURE_OPENAI_ENDPOINT"],
        temperature=0,
    )


def extract_llm_span(trace: dict) -> dict:
    """Pick the LLM span from the trace (assumes one — adjust if more)."""
    spans = trace.get("spans") or []
    for s in spans:
        if s.get("type") == "LLM":
            return s
    raise SystemExit("No LLM span found in this trace.")


def parse_input_payload(raw):
    """LLM span input may be a JSON-encoded dict like {\"question\":..., \"context\":[...]}.
    Returns (question, context_list)."""
    if isinstance(raw, str):
        try:
            raw = json.loads(raw)
        except json.JSONDecodeError:
            return raw, []          # plain string input, no context attached
    if isinstance(raw, dict):
        question = raw.get("question") or raw.get("input") or ""
        context = raw.get("context") or []
        if not isinstance(context, list):
            context = [str(context)]
        return question, context
    return str(raw), []


# ---- Run ----
trace = fetch_trace(TRACE_UUID)
llm_span = extract_llm_span(trace)

question, context = parse_input_payload(llm_span.get("input"))
answer = str(llm_span.get("output"))

print(f"Trace    : {TRACE_UUID}")
print(f"LLM span : {llm_span.get('uuid')}")
print(f"Question : {question}")
print(f"Context  : {len(context)} chunk(s)")
for c in context:
    print(f"           - {c}")
print(f"Answer   : {answer}")

judge = build_judge()

if METRIC == "summarization":
    source = "\n".join(context) if context else question
    metric = SummarizationMetric(model=judge, threshold=0.5, include_reason=True)
    metric.measure(LLMTestCase(input=source, actual_output=answer))
elif METRIC == "faithfulness":
    metric = FaithfulnessMetric(model=judge, threshold=0.5, include_reason=True)
    metric.measure(LLMTestCase(
        input=question,
        actual_output=answer,
        retrieval_context=context,
    ))
else:
    raise SystemExit(f"Unknown METRIC: {METRIC}")

print(f"\nMetric   : {type(metric).__name__}")
print(f"Score    : {metric.score:.3f}  (threshold {metric.threshold})")
print(f"Passed   : {metric.score >= metric.threshold}")
print(f"Reason   : {metric.reason}")